In [0]:
-- Business Question 3: Zone-Level Mobility Patterns & Opportunities
-- Grain: Day of week + hour + zone role + taxi zone + weather condition
-- Metrics: Trip count, fare, distance, duration
-- Period: March-May 2026
-- Limitation: Weather is aligned to each trip's pickup date and hour.
WITH base AS (
  SELECT
    d.day_of_week,
    d.day_name,
    h.hour_of_day AS hour,
    h.day_period AS time_of_day,
    CASE
      WHEN t.temperature_c < 0 THEN 'freezing'
      WHEN t.temperature_c < 10 THEN 'cool'
      WHEN t.temperature_c < 20 THEN 'mild'
      ELSE 'warm'
    END AS temperature_band,
    CASE
      WHEN t.precipitation_mm > 0 THEN TRUE
      ELSE FALSE
    END AS precipitation_flag,
    t.pickup_zone_key,
    t.dropoff_zone_key,
    t.fare_amount,
    t.trip_distance,
    t.trip_duration_minutes
  FROM
    nyc_mobility.gold.fact_trip AS t
      JOIN nyc_mobility.gold.dim_date AS d
        ON t.pickup_date_key = d.date_key
      JOIN nyc_mobility.gold.dim_hour AS h
        ON t.pickup_hour_key = h.hour_key
),
zone_activity AS (
  SELECT
    day_of_week,
    day_name,
    hour,
    time_of_day,
    temperature_band,
    precipitation_flag,
    'pickup' AS zone_role,
    pickup_zone_key AS location_id,
    fare_amount,
    trip_distance,
    trip_duration_minutes
  FROM
    base
  UNION ALL
  SELECT
    day_of_week,
    day_name,
    hour,
    time_of_day,
    temperature_band,
    precipitation_flag,
    'dropoff' AS zone_role,
    dropoff_zone_key AS location_id,
    fare_amount,
    trip_distance,
    trip_duration_minutes
  FROM
    base
),
aggregated AS (
  SELECT
    a.day_of_week,
    a.day_name,
    a.hour,
    a.time_of_day,
    a.temperature_band,
    a.precipitation_flag,
    a.zone_role,
    z.borough,
    z.zone,
    COUNT(*) AS trip_count,
    COUNT(a.fare_amount) AS fare_count,
    SUM(a.fare_amount) AS total_fare,
    COUNT(a.trip_distance) AS distance_count,
    SUM(a.trip_distance) AS total_distance,
    COUNT(a.trip_duration_minutes) AS duration_count,
    SUM(a.trip_duration_minutes) AS total_duration_minutes
  FROM
    zone_activity AS a
      JOIN nyc_mobility.gold.dim_zone AS z
        ON a.location_id = z.zone_key
  GROUP BY
    a.day_of_week,
    a.day_name,
    a.hour,
    a.time_of_day,
    a.temperature_band,
    a.precipitation_flag,
    a.zone_role,
    z.borough,
    z.zone
)
SELECT
  *,
  ROUND(
    SUM(total_distance) OVER (PARTITION BY zone_role)
      / NULLIF(SUM(distance_count) OVER (PARTITION BY zone_role), 0),
    2
  ) AS avg_trip_distance,
  ROUND(
    SUM(total_distance) OVER (PARTITION BY zone_role, borough)
      / NULLIF(SUM(distance_count) OVER (PARTITION BY zone_role, borough), 0),
    2
  ) AS avg_trip_distance_by_borough,
  ROUND(
    SUM(total_fare) OVER (PARTITION BY zone_role, borough)
      / NULLIF(SUM(fare_count) OVER (PARTITION BY zone_role, borough), 0),
    2
  ) AS avg_fare_by_borough
FROM
  aggregated;